# Objective 1c: layoff discussion on r/jobs vs layoffs.fyi

Checks whether layoff talk on r/jobs moves with reported layoffs, the external check in the proposal. Neither K-means nor LDA found a separate layoff topic, so the r/jobs side is measured with keywords ("laid off" and "layoff(s)", rules in `functions/layoff_mentions.py`):
- **Full archive**: every r/jobs comment in the Arctic Shift archive that matches, collected by `scripts/collect_layoff_mentions.py`, per 1,000 comments in the subreddit that month.
- **Part 2 sample**: the same rule on the 1,000-comment monthly sample, to check that the sample agrees with the full archive.
- **LDA T1** (K = 10, job-search difficulty, hiring process and AI): the topic that layoff comments lean towards most, as a reference curve.

The external side is layoffs.fyi (Kaggle mirror `swaptr/layoffs-2022`): reported layoff events and employees laid off per month. Only the 9 complete months, January–September 2026, are compared, so correlations are descriptive.

## 1. Setup

In [ ]:
# Add the repo root to Python's search path
import sys
sys.path.append('..')

from functions.layoff_mentions import layoff_terms

import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

MENTIONS_DIR = Path('../dataset/reddit/layoff_mentions')
LAYOFFS_FILE = Path('../dataset/layoffs/layoffs.csv')
TOPICS_DIR = Path('../dataset/monthly_Topics')
FIG_DIR = MENTIONS_DIR / 'figures'
TERMS = ['laid_off', 'layoff']
COMPLETE = [f'2026-{m:02d}' for m in range(1, 10)]

## 2. Layoff mentions in the full r/jobs archive
For each month: the archive's total r/jobs comments, the comments the archive search returned for each term, and those that pass the exact rule. A comment that uses both terms counts once. Precision (exact / returned) shows how loose the archive search is: "laid off" also returns "laid back" and "laid out". October is cut off on 28 October, so it has no rate.

In [ ]:
run = json.loads((MENTIONS_DIR / 'run.json').read_text(encoding='utf-8'))
counts = pd.read_csv(MENTIONS_DIR / 'monthly_counts.csv').set_index('month')
counts['per_1000'] = counts['any_term_exact'] / counts['archive_comments'] * 1000
counts.loc[counts['partial_month'], 'per_1000'] = np.nan
for term in TERMS:
    counts[f'{term}_precision'] = counts[f'{term}_exact'] / counts[f'{term}_hits']
print('Collection finished', run['finished_utc'], '| archive queries:', run['queries'])
counts[['archive_comments', 'laid_off_exact', 'layoff_exact', 'any_term_exact', 'per_1000',
        'laid_off_precision', 'layoff_precision']].round(3)

## 3. Does Part 2's sample agree with the full archive?
The same rule on the 1,000 sampled comments per month, with a 95% Wilson interval. If the archive rate falls inside most intervals, the sample is a fair picture of the subreddit for this kind of content. The sample excludes deleted, bot and very short comments; the archive totals do not, which pulls the archive rate slightly down.

In [ ]:
corpus = pd.read_csv(TOPICS_DIR / 'comments_clean.csv', dtype={'comment_id': str}, keep_default_na=False)
corpus['mentions'] = corpus['text'].map(lambda text: bool(layoff_terms(text, TERMS)))
sample = corpus.groupby('month')['mentions'].agg(['sum', 'size'])


def wilson(k, n, z=1.96):
    p = k / n
    centre = (p + z ** 2 / (2 * n)) / (1 + z ** 2 / n)
    half = z * np.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2)) / (1 + z ** 2 / n)
    return centre - half, centre + half


low, high = wilson(sample['sum'], sample['size'])
check = pd.DataFrame({
    'sample_mentions': sample['sum'],
    'sample_per_1000': sample['sum'] / sample['size'] * 1000,
    'sample_low': low * 1000, 'sample_high': high * 1000,
    'archive_per_1000': counts['per_1000'],
})
check['inside_interval'] = check['archive_per_1000'].between(check['sample_low'], check['sample_high'])
print(f"Archive rate inside the sample interval in {check.loc[COMPLETE, 'inside_interval'].sum()} of {len(COMPLETE)} complete months")
check.round(1)

## 4. layoffs.fyi
Reported layoff events, distinct companies and employees laid off per month, overall and for the United States, where most r/jobs users work. Many events have no headcount, and a few very large cuts dominate some months (largest events below), so the number of events is the steadier measure.

In [ ]:
layoffs = pd.read_csv(LAYOFFS_FILE)
layoffs['date'] = pd.to_datetime(layoffs['date'], format='%m/%d/%Y')
layoffs = layoffs[layoffs['date'] >= '2026-01-01'].assign(month=lambda d: d['date'].dt.strftime('%Y-%m'))
us = layoffs['country'] == 'United States'
by_month = layoffs.groupby('month')
fyi = pd.DataFrame({
    'events': by_month.size(),
    'companies': by_month['company'].nunique(),
    'laid_off': by_month['total_laid_off'].sum(),
    'missing_headcount': by_month['total_laid_off'].apply(lambda s: s.isna().mean()),
    'us_events': layoffs[us].groupby('month').size(),
    'us_laid_off': layoffs[us].groupby('month')['total_laid_off'].sum(),
})
print('Latest event in the file:', layoffs['date'].max().date())
display(fyi.round(2))
layoffs.nlargest(5, 'total_laid_off')[['company', 'date', 'total_laid_off', 'country', 'industry']]

## 5. Comparison
Spearman correlations over the 9 complete months, for the same month and with r/jobs one month later than layoffs.fyi (8 pairs). With so few points, |ρ| has to reach about 0.7 before it is distinguishable from chance, and 18 correlations are computed, so one or two with p < 0.05 are expected by chance alone. Read the table as a description of how the curves move, not as evidence of a relationship. Raw mention counts also follow the overall fall in r/jobs activity, so compare the rate per 1,000 comments rather than counts. In the figure, every curve is standardised (z-score over January–September) so they share one axis.

In [ ]:
lda_shares = pd.read_csv(TOPICS_DIR / 'lda' / 'lda_monthly_shares.csv')
lda_t1 = lda_shares[lda_shares['topic'] == 1].set_index('month')['seed_42'] * 100

series = pd.DataFrame({
    'r/jobs mentions per 1,000 comments': counts['per_1000'],
    'r/jobs mentions': counts['any_term_exact'],
    'LDA T1 share (%)': lda_t1,
    'layoffs.fyi events': fyi['events'],
    'layoffs.fyi employees laid off': fyi['laid_off'],
    'layoffs.fyi US events': fyi['us_events'],
}).loc[COMPLETE]
display(series.round(2))

reddit = ['r/jobs mentions per 1,000 comments', 'r/jobs mentions', 'LDA T1 share (%)']
external = ['layoffs.fyi events', 'layoffs.fyi employees laid off', 'layoffs.fyi US events']
rows = []
for r in reddit:
    for e in external:
        same = spearmanr(series[r], series[e])
        lagged = spearmanr(series[r].iloc[1:].to_numpy(), series[e].iloc[:-1].to_numpy())
        rows.append({'r/jobs': r, 'layoffs.fyi': e,
                     'rho_same_month': same.statistic, 'p_same_month': same.pvalue,
                     'rho_rjobs_1_month_later': lagged.statistic, 'p_1_month_later': lagged.pvalue})
correlations = pd.DataFrame(rows)
correlations.round(3)

In [ ]:
FIG_DIR.mkdir(parents=True, exist_ok=True)
z = (series - series.mean()) / series.std()
x = range(len(COMPLETE))
ticks = [m[5:] for m in COMPLETE]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for column, style in [('r/jobs mentions per 1,000 comments', '-o'), ('LDA T1 share (%)', ':s'),
                      ('layoffs.fyi events', '--^'), ('layoffs.fyi employees laid off', '-.d')]:
    axes[0].plot(x, z[column], style, label=column)
axes[0].axhline(0, color='grey', linewidth=0.5)
axes[0].set_xticks(list(x), ticks)
axes[0].set(xlabel='2026 month', ylabel='z-score (Jan–Sep)', title='Layoff talk on r/jobs vs reported layoffs')
axes[0].legend(fontsize=8)

# both axes start at 0 so the fall in layoff mentions is not exaggerated
volume = counts.loc[COMPLETE, 'archive_comments'] / 1000
mentions = counts.loc[COMPLETE, 'any_term_exact']
axes[1].bar(x, volume, color='#c9d6e8', label='all r/jobs comments (thousands)')
axes[1].set(xlabel='2026 month', ylabel='All comments (thousands)', ylim=(0, volume.max() * 1.35),
            title='r/jobs volume and layoff mentions (full archive)')
axes[1].set_xticks(list(x), ticks)
twin = axes[1].twinx()
twin.plot(x, mentions, '-o', color='#d4553b', label='comments mentioning layoffs')
twin.set(ylabel='Comments mentioning layoffs', ylim=(0, mentions.max() * 1.35))
bars, bar_labels = axes[1].get_legend_handles_labels()
line, line_labels = twin.get_legend_handles_labels()
axes[1].legend(bars + line, bar_labels + line_labels, loc='upper right', fontsize=8)
plt.tight_layout()
plt.savefig(FIG_DIR / 'layoffs_comparison.png', dpi=150)
plt.show()

fig, ax = plt.subplots(figsize=(9, 4))
ax.errorbar(x, check.loc[COMPLETE, 'sample_per_1000'],
            yerr=[check.loc[COMPLETE, 'sample_per_1000'] - check.loc[COMPLETE, 'sample_low'],
                  check.loc[COMPLETE, 'sample_high'] - check.loc[COMPLETE, 'sample_per_1000']],
            fmt='o', capsize=4, label='Part 2 sample (95% interval)')
ax.plot(x, check.loc[COMPLETE, 'archive_per_1000'], '-s', label='full archive')
ax.set_xticks(list(x), ticks)
ax.set(xlabel='2026 month', ylabel='Mentions per 1,000 comments', title='Sample vs full archive')
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / 'sample_vs_archive.png', dpi=150)
plt.show()

## 6. Export
Writes the monthly table, the sample check and the correlations to `dataset/reddit/layoff_mentions/` (ignored by git), next to the collected counts.

In [ ]:
series.join(fyi[['companies', 'missing_headcount', 'us_laid_off']]).to_csv(MENTIONS_DIR / 'comparison_monthly.csv')
check.to_csv(MENTIONS_DIR / 'sample_vs_archive.csv')
correlations.to_csv(MENTIONS_DIR / 'comparison_correlations.csv', index=False)
print('Saved to', MENTIONS_DIR)

## 7. Notes
- layoffs.fyi mostly tracks tech companies and dates layoffs by announcement; r/jobs covers every industry and the time people are affected. A weak link between the two is therefore not surprising.
- Only two terms are counted. "severance", "downsizing" and "redundancy" were left out because they are ambiguous (severance also comes up when people resign) or kept timing out in the archive, and "laying off" is not covered.
- The archive is a public copy of Reddit and its totals include deleted and bot comments.
- Nine months of data cannot separate a real relationship from chance; report the curves and correlations as descriptive.